# 03 — Train ALL 13 Models
LR, KNN, GNB, DT, RF, ExtraTrees, AdaBoost, GB, XGB, LGB, CatBoost, MLP, SGD

**Optimized:** tqdm, reduced estimators, 3-fold CV, no slow SVM

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, pickle, os, time
from tqdm.notebook import tqdm
if os.path.basename(os.getcwd())=='notebooks': os.chdir('..')
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
SEED=42; PROC='data/processed'; MOD='models'
os.makedirs(f'{MOD}/model_zoo',exist_ok=True)
os.makedirs('charts/model_comparison',exist_ok=True)
print('Ready')

## 1. Load Data

In [ ]:
X=pd.read_csv(f'{PROC}/X_processed.csv')
y=pd.read_csv(f'{PROC}/y_target.csv').iloc[:,0]
CLASS_NAMES=pickle.load(open(f'{MOD}/class_names.pkl','rb'))
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
print(f'Train:{X_train.shape[0]:,} Test:{X_test.shape[0]:,}')

## 2. Define Models (Optimized)

In [ ]:
def get_models():
    return {
        'logistic_regression': LogisticRegression(C=1,max_iter=1000,solver='lbfgs',class_weight='balanced',random_state=SEED,n_jobs=-1),
        'knn': KNeighborsClassifier(n_neighbors=7,weights='distance',n_jobs=-1),
        'gaussian_nb': GaussianNB(),
        'sgd_classifier': SGDClassifier(loss='log_loss',alpha=0.001,class_weight='balanced',random_state=SEED,n_jobs=-1),
        'decision_tree': DecisionTreeClassifier(max_depth=10,class_weight='balanced',random_state=SEED),
        'random_forest': RandomForestClassifier(n_estimators=100,max_depth=12,class_weight='balanced',random_state=SEED,n_jobs=-1),
        'extra_trees': ExtraTreesClassifier(n_estimators=100,max_depth=12,class_weight='balanced',random_state=SEED,n_jobs=-1),
        'adaboost': AdaBoostClassifier(n_estimators=80,learning_rate=0.1,random_state=SEED),
        'gradient_boosting': GradientBoostingClassifier(n_estimators=80,learning_rate=0.1,max_depth=5,random_state=SEED),
        'xgboost': XGBClassifier(n_estimators=100,learning_rate=0.1,max_depth=6,eval_metric='mlogloss',use_label_encoder=False,random_state=SEED,n_jobs=-1,tree_method='hist'),
        'lightgbm': LGBMClassifier(n_estimators=100,learning_rate=0.1,num_leaves=31,class_weight='balanced',random_state=SEED,verbose=-1,n_jobs=-1),
        'catboost': __import__('catboost').CatBoostClassifier(iterations=100,learning_rate=0.1,depth=6,random_seed=SEED,verbose=0),
        'mlp': MLPClassifier(hidden_layer_sizes=(64,32),activation='relu',solver='adam',max_iter=300,early_stopping=True,random_state=SEED),
    }
models=get_models()
print(f'{len(models)} models defined')

## 3. Cross-Validate & Train All (tqdm)

In [ ]:
cv=StratifiedKFold(3,shuffle=True,random_state=SEED)
cv_res={}; trained={}

for name,model in tqdm(models.items(),desc='Training Models'):
    t=time.time()
    try:
        scores=cross_val_score(model,X_train,y_train,cv=cv,scoring='f1_macro',n_jobs=-1)
        model.fit(X_train,y_train)
        elapsed=time.time()-t
        cv_res[name]={'f1':round(scores.mean(),4),'std':round(scores.std(),4),'time':round(elapsed,2)}
        trained[name]=model
        tqdm.write(f'  ✓ {name}: F1={scores.mean():.4f}+/-{scores.std():.4f} ({elapsed:.1f}s)')
    except Exception as e:
        tqdm.write(f'  ✗ {name}: FAIL - {e}')
        cv_res[name]={'f1':0,'std':0,'time':0}

pd.DataFrame(cv_res).T.sort_values('f1',ascending=False)

## 4. CV Results Chart

In [ ]:
import matplotlib.pyplot as plt
cv_df=pd.DataFrame(cv_res).T.sort_values('f1')
fig,ax=plt.subplots(figsize=(10,7))
ax.barh(cv_df.index,cv_df['f1'],color=plt.cm.viridis(np.linspace(0.2,0.8,len(cv_df))),edgecolor='black')
for i,(idx,row) in enumerate(cv_df.iterrows()):
    ax.text(row['f1']+0.005,i,f'{row["f1"]:.4f}',va='center',fontsize=9)
ax.set_title('CV F1 Score — All Models',fontweight='bold'); ax.set_xlim(0,1)
plt.tight_layout(); plt.savefig('charts/model_comparison/07_cv_results.png',dpi=150,bbox_inches='tight'); plt.show()

## 5. Save Models

In [ ]:
for name,model in tqdm(trained.items(),desc='Saving Models'):
    pickle.dump(model,open(f'{MOD}/model_zoo/{name}.pkl','wb'))
pickle.dump(cv_res,open(f'{MOD}/cv_results.pkl','wb'))
print(f'Saved {len(trained)} models to {MOD}/model_zoo/')